# Exploratory Data Analysis & Brand Selection
## Hiver SDE Intern Take-Home: AI Twitter Support Agent

This notebook explores the Kaggle Customer Support on Twitter dataset (	houghtvector/customer-support-on-twitter), analyzes top corporate support brands across multiple operational dimensions, defends the final brand selection, and derives an empirical intent taxonomy via semantic clustering.


In [ ]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt

# Load pre-computed brand summary from the 2.81M tweet dataset
with open('../data/processed/eda_brand_summary.json', 'r') as f:
    summary = json.load(f)

print(f"Total Tweets in Dataset: {summary['total_rows']:,}")
top_brands_df = pd.DataFrame(list(summary['top_brands'].items()), columns=['Brand', 'Outbound Tweets'])
top_brands_df.head(10)


In [ ]:
# Visualize Top 10 Support Brands by Volume
top_10 = top_brands_df.head(10)
print(top_10.to_string(index=False))


### Candidate Brand In-Depth Comparison
We evaluate the top candidates across four critical criteria:
1. **Outbound reply volume** (ensuring sufficient data depth)
2. **Matched inbound inquiries** (pairs of customer issue -> brand reply)
3. **English language ratio** (filtering non-English support traffic)
4. **Average conversation thread depth** (complexity of interaction)


In [ ]:
with open('../data/processed/candidate_brands_metrics.json', 'r') as f:
    candidates_metrics = json.load(f)

candidates_df = pd.DataFrame.from_dict(candidates_metrics, orient='index')
candidates_df.index.name = 'Brand'
candidates_df.reset_index(inplace=True)
print(candidates_df.to_string(index=False))


### Selected Brand: AppleSupport

#### Explicit Selection Defense:
1. **Usable Grounding Corpus Volume:** With **106,860 brand replies** and **106,648 matched customer inbound pairs**, AppleSupport comfortably satisfies the threshold of $\ge 5k$ usable resolved threads even after conservative subsampling and strict train/eval splits.
2. **Single, Consistent Brand Voice:** Unlike AmazonHelp (which acts as a multi-seller marketplace mediating third-party merchant disputes, varied courier contracts, and regional marketplace terms), AppleSupport speaks in a singularly unified, professional, highly consistent tone ("We're here to help...", "Let's work together to get this sorted out", standard privacy & diagnostic protocols).
3. **Bounded-yet-Diverse Issue Surface:** Apple's support envelope covers a broad yet well-defined taxonomy (iOS system updates, battery/hardware health, iCloud & Apple ID authentication, App Store billing/subscriptions, Bluetooth/AirPods connectivity, network settings) without degenerating into trivial single-intent loops (like airline delay tracking) or boundless open-domain ambiguity.
4. **Data Cleanliness & English Homogeneity:** 99.6% of AppleSupport inbound inquiries in this dataset are English, with an average thread depth of 1.99 (clean 2-turn problem-to-resolution arcs ideal for customer support triage and drafting).


### Subsampling Strategy & Budget Defense
To strictly satisfy the **under 15-minute runtime budget on a consumer laptop without a GPU**, we cannot embed or classify the full 106,000 AppleSupport threads.
- **Subsample Size:** We select  = 25,000$ inbound-reply pairs for the brand grounding corpus.
- **Time Stratification:** Tweets are sampled proportionally across active months in the dataset. This prevents seasonality bias or single-event distortions (e.g. the major iOS 11 battery throttling controversy spike in late 2017) from skewing the historical representation.
- **Split Breakdown:**
  - Grounding Retrieval Corpus: 20,000 resolved pairs
  - Train/Validation Pool (for baseline classifiers): 4,800 pairs
  - Golden Evaluation Test Split: 200 held-out, human-validated pairs (completely excluded from retrieval index)


### Intent Taxonomy Derivation

To avoid inventing intents in the abstract, we inspect semantic clusters of customer inbound queries.
The 8 derived core intents are:
1. os_system_update: Issues installing, updating, or crashing on iOS/macOS versions.
2. attery_power_hardware: Rapid battery drain, device overheating, physical hardware defects.
3. ccount_appleid_icloud: Apple ID login lockouts, password resets, iCloud backup/storage errors.
4. illing_subscription_appstore: Unexpected charges, subscription cancellations, App Store refund requests.
5. udio_bluetooth_connectivity: AirPods pairing, speaker distortion, Bluetooth drops.
6. 
etwork_wifi_cellular: No cellular service, Wi-Fi dropping, carrier SIM errors.
7. usage_feature_settings: Questions on how to configure settings, camera features, or native apps.
8. general_complaint_feedback: Dissatisfaction with brand, store experience, or general sentiment without a specific bug.
9. other: Ambiguous, uninterpretable, or out-of-domain inquiries.
